# Paso 7 — Modelo de recomendación

Notebook de verificación del motor determinista del **score de compatibilidad** (decisión A7,
AGENTS.md): filtros duros de alergia/dieta, D1 con signo (5 nutrientes de dirección fija), D3 de
preferencias, pesos desde el orden de 3 prioridades (A6), la regla de cobertura `cov` (A2) y el
ensamblado final con explicación rica (A10).

Mismo estilo que `03_transformacion_score.ipynb`: cada módulo de `src/nutrimatch/engine/` se
verifica aquí contra los datos reales del snapshot `off_csv_20260919`, con 2-3 perfiles de usuario
de ejemplo. Las cifras que aparecen en las celdas de markdown se transcriben de la salida real de
las celdas de código inmediatamente anteriores, después de ejecutar el notebook — no son
estimaciones.

**Alcance de esta versión (ver plan del paso 7)**: D1 solo puntúa 5 nutrientes con signo fijo y
universal (azúcares−, sal−, grasa saturada−, fibra+, proteína+). `energy-kcal_100g`, `fat_100g` y
`carbohydrates_100g` se siguen calculando y se muestran, pero no puntúan todavía (el documento
maestro los deja "según meta" sin definir esa meta en ningún lugar del proyecto).

In [1]:
import math
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)

from nutrimatch.engine.compatibility_score import calcular_score_compatibilidad
from nutrimatch.engine.constants import CORE8_NUTRIENTES
from nutrimatch.engine.coverage import UMBRAL_COV_INFORMACION_INSUFICIENTE
from nutrimatch.engine.hard_filters import evaluar_alergia, evaluar_dieta
from nutrimatch.engine.nutrition_score import NUTRIENTES_D1_SIGNO, calcular_subpuntaje_d1
from nutrimatch.engine.preference_score import calcular_d3
from nutrimatch.engine.product_naming import resolver_nombre_producto
from nutrimatch.engine.user_weights import convertir_prioridades_a_pesos

SNAPSHOT_ID = "off_csv_20260919"
RUTA_PROCESADOS = Path("../datos/procesados")
RUTA_SNAPSHOT_CRUDO = Path("../datos/procesados/off_mexico_20260919.parquet")

con = duckdb.connect()

ruta_matriz = RUTA_PROCESADOS / "matriz_nut_100g_20260919.parquet"
matriz = con.execute(f"SELECT * FROM '{ruta_matriz}'").df()

# Columnas extra del snapshot crudo, no incluidas en matriz_nut_100g (paso 6): necesarias para
# los filtros duros (allergens, traces, ingredients_analysis_tags), D3 (labels_tags) y el nombre
# a mostrar (product_name, generic_name, abbreviated_product_name — ver decisión A28).
extra = con.execute(
    f"""
    SELECT code, product_name, generic_name, abbreviated_product_name,
           labels_tags, allergens, traces, ingredients_analysis_tags
    FROM '{RUTA_SNAPSHOT_CRUDO}'
    """
).df()

df = matriz.merge(extra, on="code", how="left", validate="one_to_one")

# Decisión A28: product_name puede venir vacío en el export CSV aunque el producto SÍ tenga
# generic_name o abbreviated_product_name (hallazgo original de esta sección). Se resuelve con
# fallback entre las tres columnas del propio OFF, preservando el crudo en product_name_bruto.
df = df.rename(columns={"product_name": "product_name_bruto"})
nombres_resueltos = df.apply(
    lambda r: resolver_nombre_producto(
        r["product_name_bruto"], r["generic_name"], r["abbreviated_product_name"]
    ),
    axis=1,
)
df["product_name"] = nombres_resueltos.apply(lambda t: t[0])
df["product_name_flag_respaldo_usado"] = nombres_resueltos.apply(lambda t: t[1])

print("Filas en matriz_nut_100g:", len(matriz))
print("Filas en columnas extra:", len(extra))
print("Filas tras el merge:", len(df))
print("universo_puntuable (A22):", int(df["universo_puntuable"].sum()))
print(f"product_name resuelto por respaldo (A28): {df['product_name_flag_respaldo_usado'].sum()} productos")
df[
    ["code", "product_name_bruto", "product_name", "product_name_flag_respaldo_usado", "categoria_referencia", "d2"]
].head(3)

Filas en matriz_nut_100g: 16851
Filas en columnas extra: 16851
Filas tras el merge: 16851
universo_puntuable (A22): 5864
product_name resuelto por respaldo (A28): 68 productos


,code,product_name_bruto,product_name,product_name_flag_respaldo_usado,categoria_referencia,d2
0,00000140,tisane saveur lunaire,tisane saveur lunaire,False,en:herbal-teas,25.000000
1,00000154,Mct Oil Powder,Mct Oil Powder,False,NaN,22.222222
2,00000285,NaN,NaN,False,NaN,NaN


**Nota (decisión A28)**: al revisar el ranking de la Sección 7 se detectó que `product_name` viene
vacío para algunos productos que sí tienen nombre en `generic_name` o `abbreviated_product_name`
(ambos campos del propio export de OFF, no un dato externo) — por ejemplo el code `5060323907641`,
sin `product_name` pero con `generic_name = "Organic Smooth Almond Butter"`. Se resuelve con
`nutrimatch.engine.product_naming.resolver_nombre_producto` (fallback
`product_name → generic_name → abbreviated_product_name`), preservando el valor crudo en
`product_name_bruto` y marcando con `product_name_flag_respaldo_usado` los casos donde se usó un
campo de respaldo — mismo patrón de trazabilidad que la corrección de escala de sal (A21). El
resultado de la celda anterior muestra cuántos productos del universo necesitaron este respaldo.

## Sección 1 — Perfiles de usuario de ejemplo

Tres perfiles pensados para ejercitar rutas distintas del motor:

- **Ana**: alérgica a gluten, dieta vegana, valora `en:organic` y `en:no-gluten`, prioriza
  nutrición > preferencias > procesamiento.
- **Beto**: sin alergias ni dieta declarada, valora `en:fair-trade`, prioriza procesamiento >
  nutrición > preferencias.
- **Caro**: alérgica a leche y soya, dieta vegetariana, **no eligió ninguna etiqueta valorada**
  (ejercita el caso D3 = "sin dato" por ausencia de preferencias, no por falta de `labels_tags`),
  prioriza preferencias > nutrición > procesamiento.

`perfil_base` (Equilibrado / Salud máxima / Económico / Mínimo procesado) es un concepto de
UI/dominio que fijaría el orden de prioridades por defecto de cada perfil; aquí se pasa
directamente el orden ya resuelto, como especifica el plan del paso 7.

In [2]:
PERFILES = {
    "Ana": {
        "alergenos_declarados": ["en:gluten"],
        "dieta_declarada": "vegano",
        "etiquetas_valoradas": ["en:organic", "en:no-gluten"],
        "orden_prioridades": ["D1", "D3", "D2"],
    },
    "Beto": {
        "alergenos_declarados": [],
        "dieta_declarada": None,
        "etiquetas_valoradas": ["en:fair-trade"],
        "orden_prioridades": ["D2", "D1", "D3"],
    },
    "Caro": {
        "alergenos_declarados": ["en:milk", "en:soybeans"],
        "dieta_declarada": "vegetariano",
        "etiquetas_valoradas": [],
        "orden_prioridades": ["D3", "D1", "D2"],
    },
}

pd.DataFrame(PERFILES).T

,alergenos_declarados,dieta_declarada,etiquetas_valoradas,orden_prioridades
Ana,[en:gluten],vegano,"[en:organic, en:no-gluten]","[D1, D3, D2]"
Beto,[],None,[en:fair-trade],"[D2, D1, D3]"
Caro,"[en:milk, en:soybeans]",vegetariano,[],"[D3, D1, D2]"


## Sección 2 — Pesos desde el orden de prioridades (decisión A6)

**Propuesta del paso 7, a verificar aquí**: pesos proporcionales a una secuencia fija 3/2/1 según
la posición (1º lugar = 3 puntos, 2º = 2, 3º = 1), normalizada para sumar 1
(`nutrimatch.engine.user_weights.convertir_prioridades_a_pesos`). Dado que solo hay 3! = 6
permutaciones posibles de 3 elementos, los pesos resultantes son siempre los mismos tres valores
(≈0,50 / 0,33 / 0,17) repartidos según el orden — no hay parámetros libres que ajustar, solo
verificar que el reparto se sienta razonable y transparente para explicar en la interfaz ("sin
controles numéricos directos", A6).

In [3]:
pesos_por_perfil = {
    nombre: convertir_prioridades_a_pesos(perfil["orden_prioridades"])
    for nombre, perfil in PERFILES.items()
}
for nombre, perfil in PERFILES.items():
    PERFILES[nombre]["pesos"] = pesos_por_perfil[nombre]

pd.DataFrame(pesos_por_perfil).T.round(4)

,D1,D3,D2
Ana,0.5000,0.3333,0.1667
Beto,0.3333,0.1667,0.5000
Caro,0.3333,0.5000,0.1667


## Sección 3 — Filtros duros de alergia y dieta (decisiones A1, A15)

Se aplican sobre el **universo completo** (16.851 productos), no solo el puntuable: los filtros
duros son independientes del score y deben poder mostrarse incluso para un producto sin
percentiles calculables (A19 — el catálogo entero es buscable).

`evaluar_alergia` usa `allergens` y `traces`; `evaluar_dieta` usa `ingredients_analysis_tags`.
Ambos con tres estados, nunca un binario fail-safe (A15).

In [4]:
resultados_filtros = {}

for nombre, perfil in PERFILES.items():
    estado_alergia = df.apply(
        lambda r, decl=perfil["alergenos_declarados"]: evaluar_alergia(r["allergens"], r["traces"], decl),
        axis=1,
    )
    df[f"alergia_{nombre}"] = estado_alergia
    resumen = {"alergia": estado_alergia.value_counts().to_dict()}

    if perfil["dieta_declarada"] is not None:
        estado_dieta = df.apply(
            lambda r, dieta=perfil["dieta_declarada"]: evaluar_dieta(r["ingredients_analysis_tags"], dieta),
            axis=1,
        )
        df[f"dieta_{nombre}"] = estado_dieta
        resumen["dieta"] = estado_dieta.value_counts().to_dict()
    else:
        df[f"dieta_{nombre}"] = "compatible"  # sin restricción declarada: no hay nada que evaluar

    resultados_filtros[nombre] = resumen

for nombre, resumen in resultados_filtros.items():
    print(f"--- {nombre} ---")
    for filtro, conteos in resumen.items():
        total = sum(conteos.values())
        partes = ", ".join(f"{estado}: {n} ({n * 100 / total:.1f}%)" for estado, n in conteos.items())
        print(f"  {filtro}: {partes}")
    print()

--- Ana ---
  alergia: no_verificable: 13362 (79.3%), apto: 1778 (10.6%), no_apto: 1711 (10.2%)
  dieta: no_verificable: 12946 (76.8%), incompatible: 2988 (17.7%), compatible: 917 (5.4%)

--- Beto ---
  alergia: apto: 16851 (100.0%)

--- Caro ---
  alergia: no_verificable: 13362 (79.3%), no_apto: 2472 (14.7%), apto: 1017 (6.0%)
  dieta: no_verificable: 14792 (87.8%), compatible: 1192 (7.1%), incompatible: 867 (5.1%)



**Resultado**: para Ana (alérgica a gluten) y Caro (alérgica a leche y soya), "no_verificable" es el
estado más común (79,3 % en ambos casos, idéntico porque depende solo de si `allergens`/`traces`
tienen algún dato, no de qué alérgeno se busque). Esto confirma la razón de ser de A15: con el
binario original, ese 79,3 % habría caído en "no apto" y habría dejado la inmensa mayoría del
catálogo marcada como no apta para cualquier alergia, por ausencia de dato, no por presencia real
del alérgeno. Beto (sin alergias declaradas) es 100 % "apto" trivialmente — no hay nada que
verificar. En dieta, "no_verificable" también domina (76,8-87,8 %): `ingredients_analysis_tags`
tiene alta cobertura general (Sección 1 del EDA), pero la mayoría de esos tags son
`*-status-unknown` o `maybe-*`, no una afirmación positiva o negativa.

## Sección 4 — D1: nutrición con signo (decisión A7, alcance del paso 7)

D1 es **el mismo para las tres usuarias** (no depende del perfil, a diferencia de D3 y los
filtros): usa los percentiles ya calculados en el paso 6, agnósticos del usuario, y les aplica el
signo fijo de `NUTRIENTES_D1_SIGNO`. Se calcula una sola vez para todo el universo.

In [5]:
print("Nutrientes con signo en D1 v1:", NUTRIENTES_D1_SIGNO)

d1_resultados = df.apply(
    lambda r: calcular_subpuntaje_d1({n: r[f"percentil_{n}"] for n in CORE8_NUTRIENTES}),
    axis=1,
)
df["d1"] = d1_resultados.apply(lambda t: t[0])
df["d1_detalle"] = d1_resultados.apply(lambda t: t[1])

cobertura_d1 = df["d1"].notna().mean() * 100
print(f"\nD1 disponible: {df['d1'].notna().sum()} de {len(df)} productos ({cobertura_d1:.1f}%)")
df["d1"].describe()

Nutrientes con signo en D1 v1: {'sugars_100g': -1, 'salt_100g': -1, 'saturated-fat_100g': -1, 'fiber_100g': 1, 'proteins_100g': 1}

D1 disponible: 7040 de 16851 productos (41.8%)


count    7040.000000
mean       49.758499
std        13.332122
min         0.000000
25%        41.042088
50%        49.562942
75%        58.291103
max       100.000000
Name: d1, dtype: float64

In [6]:
# Detalle de D1 para un producto de ejemplo con los 5 nutrientes de signo fijo disponibles.
ejemplo = df.loc[df["code"] == "7501030457626"].iloc[0]
print(f"{ejemplo['product_name']} (categoría de referencia: {ejemplo['categoria_referencia']})")
print(f"D1 = {ejemplo['d1']:.3f}\n")
pd.DataFrame(ejemplo["d1_detalle"]).T

Pan con Centeno (categoría de referencia: en:breads)
D1 = 52.207



,percentil,signo,disponible,contribucion
sugars_100g,44.303797,-1,True,55.696203
salt_100g,72.5,-1,True,27.5
saturated-fat_100g,60.759494,-1,True,39.240506
fiber_100g,58.139535,1,True,58.139535
proteins_100g,80.45977,1,True,80.45977


**Resultado**: D1 tiene dato en 7.040 de 16.851 productos (41,8 %) — más amplio que el
`universo_puntuable` de A22 (34,8 %) porque ese umbral exige, además, D2 calculable y al menos 4 de
los 8 percentiles de core8, no solo los 5 con signo de esta versión. "Pan con Centeno" (categoría
`en:breads`) ejemplifica el cálculo: su azúcar está en el percentil 44,3 dentro de su categoría
(mediocre, ni alto ni bajo), pero al ser "menos es mejor" su contribución es 100 − 44,3 = 55,7;
su fibra está en el percentil 58,1 y, al ser "más es mejor", contribuye tal cual. D1 = 52,21 es el
promedio simple de las 5 contribuciones, con los 5 nutrientes disponibles.

## Sección 5 — D3: preferencias (decisión A7)

D3 **sí depende del perfil** (cada usuaria valora etiquetas distintas). Caro no eligió ninguna
etiqueta valorada: para ella D3 debe ser `None` para **todos** los productos, no 0 — es el caso de
"la usuaria no declaró preferencias", distinto de "el producto no tiene las etiquetas valoradas".

In [7]:
for nombre, perfil in PERFILES.items():
    etiquetas = perfil["etiquetas_valoradas"]
    d3 = df["labels_tags"].apply(lambda lt, etq=etiquetas: calcular_d3(lt, etq))
    df[f"d3_{nombre}"] = d3
    cobertura = d3.notna().mean() * 100
    print(f"{nombre} (valora {etiquetas or '— ninguna —'}): D3 disponible en {d3.notna().sum()} "
          f"productos ({cobertura:.1f}%)")

Ana (valora ['en:organic', 'en:no-gluten']): D3 disponible en 4266 productos (25.3%)
Beto (valora ['en:fair-trade']): D3 disponible en 4266 productos (25.3%)
Caro (valora — ninguna —): D3 disponible en 0 productos (0.0%)


**Resultado**: confirma el diseño de `calcular_d3`. Ana y Beto (sí eligieron etiquetas) tienen D3
disponible en el mismo 25,3 % de productos — la cobertura de `labels_tags` en el universo, no
depende de qué etiqueta específica se valore. Caro (no eligió ninguna) tiene D3 disponible en 0 %
de los productos: no es que sus etiquetas nunca aparezcan, es que la función devuelve `None` de
inmediato cuando `etiquetas_valoradas` está vacía, antes de mirar siquiera `labels_tags` del
producto.

## Sección 6 — Ensamblado del score, `cov` y banda de "información insuficiente" (A2, A7, A10)

Se llama a `calcular_score_compatibilidad` para cada producto y cada perfil, sobre el **universo
completo** (no solo `universo_puntuable`, que es un filtro previo agnóstico del usuario — A19,
A22): `cov` es la regla fina que decide, ya con los pesos de cada usuaria, si el producto entra al
ranking (umbral 0.5, A2).

In [8]:
print(f"Umbral de información insuficiente (A2): cov < {UMBRAL_COV_INFORMACION_INSUFICIENTE}\n")

resumen_cov = {}
for nombre, perfil in PERFILES.items():
    pesos = perfil["pesos"]
    resultados = df.apply(
        lambda r, p=pesos: calcular_score_compatibilidad(
            {"D1": r["d1"], "D2": r["d2"], "D3": r[f"d3_{nombre}"]}, p
        ),
        axis=1,
    )
    df[f"score_{nombre}"] = resultados.apply(lambda r: r["score_final"])
    df[f"cov_{nombre}"] = resultados.apply(lambda r: r["cov"])
    df[f"insuficiente_{nombre}"] = resultados.apply(lambda r: r["informacion_insuficiente"])
    df[f"detalle_score_{nombre}"] = resultados

    n_insuficiente = df[f"insuficiente_{nombre}"].sum()
    n_puntuable = df[f"score_{nombre}"].notna().sum()
    resumen_cov[nombre] = {
        "cov_media": df[f"cov_{nombre}"].mean(),
        "cov_mediana": df[f"cov_{nombre}"].median(),
        "n_informacion_insuficiente": n_insuficiente,
        "pct_informacion_insuficiente": n_insuficiente * 100 / len(df),
        "n_con_score": n_puntuable,
        "pct_con_score": n_puntuable * 100 / len(df),
    }

pd.DataFrame(resumen_cov).T.round(3)

Umbral de información insuficiente (A2): cov < 0.5



,cov_media,cov_mediana,n_informacion_insuficiente,pct_informacion_insuficiente,n_con_score,pct_con_score
Ana,0.360,0.167,9517.0,56.477,7334.0,43.523
Beto,0.383,0.167,9632.0,57.160,7219.0,42.840
Caro,0.206,0.000,10974.0,65.124,5877.0,34.876


**Resultado**: la banda "información insuficiente" captura entre 56,5 % (Ana) y 65,1 % (Caro) del
universo total de 16.851 productos — coherente con que D1 (41,8 %) y D2 (40,2 %) tienen la
cobertura más baja de las tres dimensiones (Sección 4 y paso 6), y con que ningún perfil pesa
tanto una sola dimensión como para que su ausencia sea automáticamente compensable. Caro es la que
más productos pierde: al no haber elegido ninguna etiqueta valorada, D3 es `None` para el 100 % del
catálogo (Sección 5) y su peso de 0,50 sobre D3 nunca contribuye — su `cov` máxima posible es 0,5
exacto (el peso conjunto de D1 y D2), por eso su mediana de `cov` es 0,0 y no 0,17 como en los otros
dos perfiles. Esto es exactamente lo esperado de la fórmula de A26: no es un error, es la
consecuencia directa de declarar cero preferencias.

## Sección 7 — Ranking: el orden debe cambiar al reordenar las prioridades

Verificación de que los pesos realmente mueven el ranking: se toma el perfil de **Ana**
(prioridad `["D1", "D3", "D2"]`, nutrición primero) y se compara su top-10 (dentro de los
productos aptos + compatibles + con score) contra el top-10 de una usuaria idéntica en todo lo
demás pero con las prioridades **invertidas** (`["D2", "D3", "D1"]`, procesamiento primero).

In [9]:
def apta_para_ranking(fila: pd.Series, nombre_perfil: str) -> bool:
    if fila[f"alergia_{nombre_perfil}"] != "apto":
        return False
    if fila[f"dieta_{nombre_perfil}"] not in ("compatible", "no_verificable"):
        # "no_verificable" no se excluye del ranking (A15): se mostraría en su propia banda con
        # advertencia, no mezclada con "apto", pero aquí solo comparamos el ORDEN del ranking, así
        # que se incluye igual que un "apto"/"compatible" para no reducir la muestra a casi nada.
        return False
    return bool(fila[f"score_{nombre_perfil}"] == fila[f"score_{nombre_perfil}"])  # no NaN


elegibles_ana = df[df.apply(lambda r: apta_para_ranking(r, "Ana"), axis=1)].copy()
print(f"Productos elegibles para el ranking de Ana: {len(elegibles_ana)}")

top10_prioridad_original = (
    elegibles_ana.sort_values("score_Ana", ascending=False)
    .head(10)[["code", "product_name", "categoria_referencia", "d1", "d2", "d3_Ana", "score_Ana"]]
)

pesos_invertidos = convertir_prioridades_a_pesos(["D2", "D3", "D1"])
print("Pesos originales de Ana:", {k: round(v, 3) for k, v in PERFILES["Ana"]["pesos"].items()})
print("Pesos invertidos (D2 primero):", {k: round(v, 3) for k, v in pesos_invertidos.items()})

resultados_invertidos = elegibles_ana.apply(
    lambda r: calcular_score_compatibilidad(
        {"D1": r["d1"], "D2": r["d2"], "D3": r["d3_Ana"]}, pesos_invertidos
    ),
    axis=1,
)
elegibles_ana["score_invertido"] = resultados_invertidos.apply(lambda r: r["score_final"])

top10_prioridad_invertida = elegibles_ana.sort_values("score_invertido", ascending=False).head(10)[
    ["code", "product_name", "categoria_referencia", "d1", "d2", "d3_Ana", "score_invertido"]
]

codigos_originales = set(top10_prioridad_original["code"])
codigos_invertidos = set(top10_prioridad_invertida["code"])
print(f"\nCódigos en común entre ambos top-10: {len(codigos_originales & codigos_invertidos)} de 10")
print("¿El orden cambió?", codigos_originales != codigos_invertidos or not (
    top10_prioridad_original["code"].tolist() == top10_prioridad_invertida["code"].tolist()
))

Productos elegibles para el ranking de Ana: 665
Pesos originales de Ana: {'D1': 0.5, 'D3': 0.333, 'D2': 0.167}
Pesos invertidos (D2 primero): {'D2': 0.5, 'D3': 0.333, 'D1': 0.167}

Códigos en común entre ambos top-10: 7 de 10
¿El orden cambió? True


In [10]:
print("Top-10 con prioridad de Ana (nutrición primero):")
display(top10_prioridad_original)

print("\nTop-10 con prioridades invertidas (procesamiento primero):")
display(top10_prioridad_invertida)

Top-10 con prioridad de Ana (nutrición primero):


,code,product_name,categoria_referencia,d1,d2,d3_Ana,score_Ana
4359,5060323907641,Organic Smooth Almond Butter,fr:pates-a-tartiner,86.591187,100.000000,100.0,93.295594
4358,5060323907634,Organic Smooth Peanut Butter,en:peanut-butters,80.000000,100.000000,100.0,90.000000
4282,5060323900536,Organic Super Crunchy Peanut Butter,en:peanut-butters,75.600000,100.000000,100.0,87.800000
2275,07064098,Yogurt Estilo Griego Natural sin Grasa,en:greek-style-yogurts,83.662108,100.000000,NaN,87.746581
13488,7503012633769,AJONJOLI VERDE,en:cereals-and-their-products,86.160410,NaN,NaN,86.160410
14299,7503024877014,organic rice thins,NaN,NaN,50.000000,100.0,83.333333
13489,7503012633776,AJONJOLI NARANJA,en:cereal-grains,76.037142,100.000000,NaN,82.027856
11138,7501518492477,Chorizo vegetariano 100% soya,en:plant-based-foods,100.000000,22.222222,NaN,80.555556
16682,8413700003264,Perejil molido,en:groceries,72.718743,100.000000,NaN,79.539058
10046,7501077400050,Harina de Maíz Nixtamalizado,en:cereal-flours,51.735136,100.000000,100.0,75.867568



Top-10 con prioridades invertidas (procesamiento primero):


,code,product_name,categoria_referencia,d1,d2,d3_Ana,score_invertido
4359,5060323907641,Organic Smooth Almond Butter,fr:pates-a-tartiner,86.591187,100.0,100.0,97.765198
4358,5060323907634,Organic Smooth Peanut Butter,en:peanut-butters,80.000000,100.0,100.0,96.666667
4282,5060323900536,Organic Super Crunchy Peanut Butter,en:peanut-butters,75.600000,100.0,100.0,95.933333
2275,07064098,Yogurt Estilo Griego Natural sin Grasa,en:greek-style-yogurts,83.662108,100.0,NaN,95.915527
13489,7503012633776,AJONJOLI NARANJA,en:cereal-grains,76.037142,100.0,NaN,94.009285
16682,8413700003264,Perejil molido,en:groceries,72.718743,100.0,NaN,93.179686
10046,7501077400050,Harina de Maíz Nixtamalizado,en:cereal-flours,51.735136,100.0,100.0,91.955856
11038,7501485800114,Amaranto Natural,en:cereal-grains,67.362699,100.0,NaN,91.840675
5361,7500478027668,Avena Quaker 3 Minutos fortificada,en:oat,64.473684,100.0,NaN,91.118421
29,0000880688789,Mister Natural - Datil Premium,en:dried-fruits,63.705548,100.0,NaN,90.926387


**Resultado**: de 665 productos elegibles para Ana (apta en alergia, no excluida por dieta,
con score calculable), el top-10 comparte 7 de 10 códigos entre ambas prioridades, pero **el orden
sí cambia** (verificado con `assert` implícito en la comparación de listas, no solo de conjuntos):
con nutrición primero, "AJONJOLI VERDE" (D2 sin dato, D1 alto) entra al top-10 en el puesto 5 pese a
no tener D2 calculable — la renormalización de A26 lo permite. Con procesamiento primero, ese mismo
producto desaparece del top-10 (su D2 ausente ya no puede compensarse con el peso que antes tenía
D1), y entran productos con D2 = 100 pero D1 más modesto, como "Amaranto Natural" (D1 = 67,36) y
"Avena Quaker 3 Minutos fortificada" (D1 = 64,47). Esto confirma que los pesos sí mueven el ranking
de forma consistente con la prioridad declarada, no son cosméticos.

## Sección 8 — Auditoría manual de la explicación (decisión A10)

A10 exige que cada resultado muestre: subpuntaje por dimensión, contribución de cada nutriente,
percentil dentro de la categoría, peso aplicado y estado de los datos. Se audita a mano el cálculo
completo de 3 productos conocidos para el perfil de **Beto** (sin alergias ni dieta, así los tres
llegan a tener score completo en vez de quedar excluidos por un filtro duro).

In [11]:
codigos_auditoria = ["7501030457626", "7501000601776", "7501011143586"]  # Pan con Centeno, Emperador Senzo, Cheetos torciditos

for codigo in codigos_auditoria:
    fila = df.loc[df["code"] == codigo].iloc[0]
    print("=" * 90)
    print(f"{fila['product_name']}  (code={codigo})")
    print(f"Categoría de referencia: {fila['categoria_referencia']} (nivel de retroceso: {fila['nivel_retroceso']})")
    print(f"Filtros duros (Beto): alergia={fila['alergia_Beto']}, dieta={fila['dieta_Beto']}")
    print()
    print("Detalle D1 (percentil, signo, contribución):")
    print(pd.DataFrame(fila["d1_detalle"]).T)
    print(f"\nD1 = {fila['d1']:.4f}")
    print(f"D2 = {fila['d2']:.4f}  (nova_group={fila['nova_group']}, additives_n={fila['additives_n']})")
    print(f"D3 (Beto, valora en:fair-trade) = {fila['d3_Beto']}  (labels_tags = {fila['labels_tags']!r})")
    print()
    pesos_beto = PERFILES["Beto"]["pesos"]
    print(f"Pesos de Beto: {({k: round(v, 4) for k, v in pesos_beto.items()})}")
    detalle_score = fila["detalle_score_Beto"]
    print(f"cov = {detalle_score['cov']:.4f}  |  informacion_insuficiente = {detalle_score['informacion_insuficiente']}")
    verificacion_manual = (
        pesos_beto["D1"] * fila["d1"] + pesos_beto["D2"] * fila["d2"] + pesos_beto["D3"] * fila["d3_Beto"]
    )
    print(f"score_final (motor)  = {detalle_score['score_final']:.4f}")
    print(f"score_final (a mano) = {verificacion_manual:.4f}")
    assert math.isclose(detalle_score["score_final"], verificacion_manual, rel_tol=1e-9)
    print()

Pan con Centeno  (code=7501030457626)
Categoría de referencia: en:breads (nivel de retroceso: 1)
Filtros duros (Beto): alergia=apto, dieta=compatible

Detalle D1 (percentil, signo, contribución):
                    percentil signo disponible contribucion
sugars_100g         44.303797    -1       True    55.696203
salt_100g                72.5    -1       True         27.5
saturated-fat_100g  60.759494    -1       True    39.240506
fiber_100g          58.139535     1       True    58.139535
proteins_100g        80.45977     1       True     80.45977

D1 = 52.2072
D2 = 13.8889  (nova_group=4, additives_n=4)
D3 (Beto, valora en:fair-trade) = 0.0  (labels_tags = 'en:low-or-no-fat,en:low-fat')

Pesos de Beto: {'D2': 0.5, 'D1': 0.3333, 'D3': 0.1667}
cov = 1.0000  |  informacion_insuficiente = False
score_final (motor)  = 24.3468
score_final (a mano) = 24.3468

Emperador Senzo  (code=7501000601776)
Categoría de referencia: en:biscuits (nivel de retroceso: 1)
Filtros duros (Beto): alergia=apt

Verificación manual de los tres productos (celda anterior): el `assert` de cada iteración compara
el `score_final` que devuelve `calcular_score_compatibilidad` contra el mismo cálculo hecho a mano
(`pesos["D1"]·D1 + pesos["D2"]·D2 + pesos["D3"]·D3`) y no lanzó ningún error — coinciden en los
tres casos (24,3468 / 19,0469 / 14,0117). Los tres productos tienen `cov = 1.0000` (D1, D2 y D3
disponibles), así que no hay renormalización que verificar aquí; esa ruta ya se ejercitó en la
Sección 6 (p. ej. Caro, que nunca supera cov = 0,5 porque D3 es `None` para ella en todos los
productos).

**Lectura de los tres casos**: los tres son NOVA=4 con D2 bajo (0,00 a 13,89), lo que domina el
score de Beto porque D2 pesa 0,50 para él (procesamiento es su prioridad #1). Emperador Senzo tiene
el D1 más bajo de los tres (40,47) por su azúcar y grasa saturada altas dentro de la categoría
`en:biscuits`; Cheetos torciditos tiene el D2 más bajo (0,00) por sus 10 aditivos, el tope
calibrado de la fórmula (A23). Ninguno de los tres tiene la etiqueta `en:fair-trade` que valora
Beto, así que D3 = 0,0 en los tres — un valor real (0 %), no un "sin dato": sus `labels_tags`
existen y ninguno coincide.

## Sección 9 — Síntesis: decisiones confirmadas en el paso 7

Verificado contra `off_csv_20260919` (16.851 productos). Estas decisiones se registran en
`AGENTS.md` como A25-A27, siguiendo el mismo patrón que A21-A24 del paso 6.

### A25. Fórmula de pesos: secuencia 3/2/1 normalizada según el orden de prioridades

Confirmado el reparto propuesto en el plan: 1º lugar = 3 puntos, 2º = 2, 3º = 1, normalizados para
sumar 1 (0,50 / 0,33 / 0,17). Con solo 3 dimensiones hay 6 permutaciones posibles; los tres valores
son siempre los mismos, solo cambia a qué dimensión se le asigna cada uno (Sección 2). Es un
reparto transparente y fácil de explicar en la interfaz, consistente con "sin controles numéricos
directos" (A6). Implementado en `nutrimatch.engine.user_weights.convertir_prioridades_a_pesos`.

### A26. Renormalización del score cuando falta una dimensión: promedio ponderado sobre las disponibles

Confirmado: cuando una dimensión no tiene dato, el score final se calcula como
`Σ wᵢ·Dᵢ / Σ wᵢ` sumando solo sobre las dimensiones con dato disponible (Sección 6, perfil Caro:
D3 es `None` en el 100 % de los productos porque no declaró etiquetas valoradas, y su score se
calcula solo con D1 y D2, renormalizado sobre `0,333 + 0,167 = 0,5`). Es coherente con que `cov` ya
mide exactamente esa fracción de peso disponible: cuando `cov ≥ 0,5` el score usa esa misma
fracción como denominador. Si `cov < 0,5` (banda "información insuficiente", A2), el score final es
`None`, nunca un número calculado sobre datos insuficientes. Implementado en
`nutrimatch.engine.compatibility_score.calcular_score_compatibilidad`.

### A27. Alcance de D1 v1: 5 nutrientes de signo fijo, 3 nutrientes informativos sin puntuar

Confirmado el alcance propuesto en el plan del paso 7: D1 puntúa únicamente azúcares, sal y grasa
saturada (signo −1, menos es mejor) y fibra y proteína (signo +1, más es mejor) —
`nutrimatch.engine.nutrition_score.NUTRIENTES_D1_SIGNO`. `energy-kcal_100g`, `fat_100g` (grasa
total) y `carbohydrates_100g` siguen calculándose y mostrándose (tienen percentil en
`matriz_nut_100g`, Sección 4 los deja fuera del detalle de D1 explícitamente), pero no puntúan: el
documento maestro los deja "según meta" sin definir en ningún lugar del proyecto qué valores toma
esa meta, y esta versión no inventa una taxonomía de objetivos que no existe. D1 tiene dato
disponible en 7.040 productos (41,8 % del universo, Sección 4) — más amplio que
`universo_puntuable` (34,8 %, A22) porque ese umbral exige además D2 calculable y ≥4 de los 8
percentiles de core8, no solo los 5 con signo.

### Nota técnica (no es una decisión metodológica): `NaN` de pandas vs `None` de Python en texto y en subpuntajes

Verificado al construir este notebook, documentado aquí para quien reutilice estos módulos: un
campo de texto ausente (`labels_tags`, `allergens`, `traces`, `ingredients_analysis_tags`) leído de
un Parquet vía DuckDB/pandas llega como `float('nan')`, no como `None`. `bool(float('nan'))` es
`True` en Python, así que un chequeo ingenuo con `not valor` no detecta la ausencia — y
`str(float('nan'))` produce el texto literal `"nan"`, que un split ingenuo por comas trataría como
un tag real. Lo mismo aplica a un subpuntaje ausente (`D1`, `D2` o `D3` como `NaN` en vez de
`None`): `NaN is not None` es `True`, así que un chequeo con `is not None` marca la dimensión como
"disponible" e infla `cov` artificialmente. `hard_filters.py`, `preference_score.py` y
`compatibility_score.py` incluyen una función `_es_texto_nulo`/`_es_nulo` que cubre ambos casos
(`None` y `NaN`), con pruebas de regresión explícitas en `tests/`.